# ApisTox 模型应用

本 Notebook 加载已经训练并验证过的模型，用于对新的 SMILES 进行预测。

注意：

- 输出概率是模型根据训练数据计算的评分，不是实验测得的毒性值；
- 模型预测反映统计关联，不能证明分子结构导致蜜蜂毒性；
- joblib 文件只能从可信来源加载。

In [1]:
from pathlib import Path

import joblib


# 兼容从仓库根目录或 notebooks 目录启动 Jupyter
CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_DIR = CURRENT_DIRECTORY.parent
else:
    PROJECT_DIR = CURRENT_DIRECTORY

MODEL_PATH = (
    PROJECT_DIR
    / "saved_models"
    / "apis_tox_morgan_random_forest.joblib"
)

# 在反序列化之前，先确认文件确实存在
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"未找到模型文件：{MODEL_PATH}")

# 只加载自己创建或来自可信来源的 joblib 文件
model_bundle = joblib.load(MODEL_PATH)

# 检查最外层对象是否符合预期
if not isinstance(model_bundle, dict):
    raise TypeError(
        f"预期模型包是字典 dict，实际得到：{type(model_bundle).__name__}"
    )

print("模型包加载成功")
print(f"文件路径：{MODEL_PATH}")
print(f"模型包类型：{type(model_bundle).__name__}")
print(f"模型包包含 {len(model_bundle)} 个字段：")

# 这里只显示字段名和数据类型，避免直接输出庞大的随机森林对象
for key, value in model_bundle.items():
    print(f"  - {key}: {type(value).__name__}")

模型包加载成功
文件路径：D:\ML\apis-tox-ml\saved_models\apis_tox_morgan_random_forest.joblib
模型包类型：dict
模型包包含 12 个字段：
  - model: Pipeline
  - classification_threshold: float
  - model_name: str
  - selection_metric: str
  - operating_goal: str
  - feature_type: str
  - feature_columns: list
  - fingerprint_parameters: dict
  - label_mapping: dict
  - training_sample_count: int
  - test_sample_count: int
  - software_versions: dict


In [2]:
# 本步骤需要使用的4个字段
required_keys = {
    "model",
    "classification_threshold",
    "fingerprint_parameters",
    "feature_columns",
}

# 检查模型包是否缺少必要字段
missing_keys = required_keys - model_bundle.keys()

if missing_keys:
    raise KeyError(f"模型包缺少必要字段：{sorted(missing_keys)}")


# 从模型包中读取对象
model = model_bundle["model"]
classification_threshold = model_bundle["classification_threshold"]
fingerprint_parameters = model_bundle["fingerprint_parameters"]
feature_columns = model_bundle["feature_columns"]


# 基本类型与数值检查
if not hasattr(model, "predict_proba"):
    raise TypeError("加载的模型不支持 predict_proba()，无法输出有毒概率")

if not isinstance(classification_threshold, (int, float)):
    raise TypeError("classification_threshold 应该是数值")

if not 0 <= classification_threshold <= 1:
    raise ValueError("分类阈值应该位于 0 到 1 之间")

if not isinstance(fingerprint_parameters, dict):
    raise TypeError("fingerprint_parameters 应该是字典")

if not isinstance(feature_columns, list):
    raise TypeError("feature_columns 应该是列表")


# 输出简洁的检查结果
print("必要字段读取成功")
print(f"模型类型：{type(model).__name__}")
print(f"分类阈值：{classification_threshold}")
print(f"Morgan参数：{fingerprint_parameters}")
print(f"特征数量：{len(feature_columns)}")
print(f"前5个特征名：{feature_columns[:5]}")
print(f"后5个特征名：{feature_columns[-5:]}")

必要字段读取成功
模型类型：Pipeline
分类阈值：0.3
Morgan参数：{'radius': 2, 'fp_size': 2048, 'include_chirality': True}
特征数量：2048
前5个特征名：['MorganBit_0', 'MorganBit_1', 'MorganBit_2', 'MorganBit_3', 'MorganBit_4']
后5个特征名：['MorganBit_2043', 'MorganBit_2044', 'MorganBit_2045', 'MorganBit_2046', 'MorganBit_2047']


In [3]:
# 从保存的参数字典中读取具体设置
radius = fingerprint_parameters["radius"]
fp_size = fingerprint_parameters["fp_size"]
include_chirality = fingerprint_parameters["include_chirality"]

# 根据指纹长度，构造模型期望的标准特征顺序
expected_feature_columns = [
    f"MorganBit_{bit_index}"
    for bit_index in range(fp_size)
]

# 检查指纹长度是否与特征列数量一致
if len(feature_columns) != fp_size:
    raise ValueError(
        f"指纹长度为 {fp_size}，"
        f"但模型保存了 {len(feature_columns)} 个特征名"
    )

# 检查特征名称及其排列顺序
if feature_columns != expected_feature_columns:
    raise ValueError("保存的Morgan特征名称或排列顺序不符合预期")

print("模型包内部一致性检查通过")
print(f"radius：{radius}")
print(f"fp_size：{fp_size}")
print(f"include_chirality：{include_chirality}")
print(f"特征顺序：{feature_columns[0]} → {feature_columns[-1]}")

模型包内部一致性检查通过
radius：2
fp_size：2048
include_chirality：True
特征顺序：MorganBit_0 → MorganBit_2047


## 输入待预测的分子

本步骤接收一个 SMILES 字符串。这里只检查输入是否为空，下一步再使用 RDKit 判断它能否被解析为分子。

In [4]:
# input() 会暂停运行，等待用户在输入框中输入内容
smiles_input = input("请输入一个SMILES：")

# strip() 删除字符串首尾可能存在的空格和换行符
smiles_input = smiles_input.strip()

# 空字符串不能进入后续的分子解析步骤
if not smiles_input:
    raise ValueError("SMILES不能为空，请重新输入")

print("已接收SMILES：")
print(smiles_input)

请输入一个SMILES： CCO


已接收SMILES：
CCO


## 检查SMILES有效性

使用 RDKit 将 SMILES 解析为分子对象。如果解析失败，则停止后续指纹计算和模型预测。

In [5]:
from rdkit import Chem


# 尝试把SMILES字符串解析为RDKit分子对象
molecule = Chem.MolFromSmiles(smiles_input)

# 解析失败时，MolFromSmiles()会返回None
if molecule is None:
    raise ValueError(
        f"无法解析SMILES：{smiles_input}\n"
        "请检查原子符号、括号、环编号和化学价是否正确"
    )

# 生成规范SMILES，便于确认RDKit实际解析出的结构
canonical_smiles = Chem.MolToSmiles(
    molecule,
    canonical=True,
    isomericSmiles=True,
)

print("SMILES解析成功")
print(f"原始SMILES：{smiles_input}")
print(f"规范SMILES：{canonical_smiles}")
print(f"原子数量：{molecule.GetNumAtoms()}")

SMILES解析成功
原始SMILES：CCO
规范SMILES：CCO
原子数量：3


## 生成Morgan指纹

使用模型包中保存的参数生成Morgan指纹，并严格按照训练时的特征列顺序整理为模型输入。

In [6]:
import numpy as np
import pandas as pd

from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator


# 使用模型包中保存的参数创建Morgan指纹生成器
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=radius,
    fpSize=fp_size,
    includeChirality=include_chirality,
)

# 为当前分子生成RDKit位指纹
morgan_fingerprint = morgan_generator.GetFingerprint(molecule)

# 创建长度为2048的全零数组
fingerprint_array = np.zeros(fp_size, dtype=np.int8)

# 把RDKit指纹转换为NumPy数组
DataStructs.ConvertToNumpyArray(
    morgan_fingerprint,
    fingerprint_array,
)

# 按模型保存的特征顺序，创建一行DataFrame
model_input = pd.DataFrame(
    [fingerprint_array],
    columns=feature_columns,
)

# 最终安全检查
if model_input.shape != (1, fp_size):
    raise ValueError(
        f"模型输入形状异常：{model_input.shape}，"
        f"预期为：(1, {fp_size})"
    )

print("Morgan指纹生成成功")
print(f"模型输入形状：{model_input.shape}")
print(f"指纹中的有效位数量：{int(fingerprint_array.sum())}")
print(f"数组中的唯一值：{np.unique(fingerprint_array)}")

Morgan指纹生成成功
模型输入形状：(1, 2048)
指纹中的有效位数量：6
数组中的唯一值：[0 1]


## 输出模型预测

使用模型输出 `label=1`（有毒类）的概率评分，再使用训练阶段确定的0.30阈值生成预测类别。

该概率是模型评分，不是实验测得的毒性值，也不能作为分子结构导致蜜蜂毒性的因果证据。

In [7]:
# 读取模型保存的类别顺序
model_classes = np.asarray(model.classes_)

# 找到“有毒类 label=1”在概率数组中的位置
toxic_class_positions = np.where(model_classes == 1)[0]

if len(toxic_class_positions) != 1:
    raise ValueError(
        f"无法唯一确定有毒类label=1的位置，"
        f"模型类别为：{model_classes}"
    )

toxic_class_index = toxic_class_positions[0]


# predict_proba()返回每个类别的预测概率
class_probabilities = model.predict_proba(model_input)[0]

# 按模型的类别顺序，提取label=1对应的概率
toxic_probability = float(
    class_probabilities[toxic_class_index]
)


# 使用训练阶段确定的0.30阈值，而不是默认的0.50
predicted_label = int(
    toxic_probability >= classification_threshold
)

predicted_class = "有毒" if predicted_label == 1 else "无毒"


print("预测完成")
print(f"模型类别顺序：{model_classes}")
print(f"有毒概率评分：{toxic_probability:.3f}")
print(f"分类阈值：{classification_threshold:.2f}")
print(f"预测标签：{predicted_label}")
print(f"预测类别：{predicted_class}")

print("\n结果说明：")
print("该概率是模型根据训练数据给出的评分，不是实验毒性值。")
print("预测结果表示统计关联，不能证明分子结构导致蜜蜂毒性。")

预测完成
模型类别顺序：[0 1]
有毒概率评分：0.249
分类阈值：0.30
预测标签：0
预测类别：无毒

结果说明：
该概率是模型根据训练数据给出的评分，不是实验毒性值。
预测结果表示统计关联，不能证明分子结构导致蜜蜂毒性。


## 基础适用域提示

使用新分子与训练集分子的最高 Morgan 指纹 Tanimoto 相似度，描述新分子与模型训练化学空间的接近程度。

相似度参考库只能包含训练集，不能包含最终测试集。

In [8]:
from pathlib import Path


# PROJECT_DIR 已在开头根据当前启动位置确定

# 只查找可能保存了样本、划分或指纹的数据文件
allowed_suffixes = {
    ".csv",
    ".parquet",
    ".npy",
    ".npz",
    ".pkl",
    ".joblib",
}

filename_keywords = {
    "sample",
    "split",
    "train",
    "fingerprint",
    "morgan",
    "feature",
}

candidate_files = []

for file_path in PROJECT_DIR.rglob("*"):
    if not file_path.is_file():
        continue

    if file_path.suffix.lower() not in allowed_suffixes:
        continue

    filename_lower = file_path.name.lower()

    if any(
        keyword in filename_lower
        for keyword in filename_keywords
    ):
        candidate_files.append(file_path)


if not candidate_files:
    raise FileNotFoundError(
        "没有找到候选文件，请回到02或03 Notebook检查保存路径"
    )

print(f"找到 {len(candidate_files)} 个候选文件：\n")

for file_path in sorted(candidate_files):
    relative_path = file_path.relative_to(PROJECT_DIR)
    file_size_kb = file_path.stat().st_size / 1024

    print(f"{relative_path}  ({file_size_kb:.1f} KB)")

找到 5 个候选文件：

processed_data\descriptor_features.csv  (84.9 KB)
processed_data\morgan_features.csv  (4173.9 KB)
processed_data\random_grouped_split.csv  (79.1 KB)
processed_data\sample_metadata.csv  (149.4 KB)
saved_models\apis_tox_morgan_random_forest.joblib  (1276.6 KB)


In [9]:
# 三个正式数据文件的路径
SAMPLE_METADATA_PATH = (
    PROJECT_DIR / "processed_data" / "sample_metadata.csv"
)

MORGAN_FEATURES_PATH = (
    PROJECT_DIR / "processed_data" / "morgan_features.csv"
)

SPLIT_PATH = (
    PROJECT_DIR / "processed_data" / "random_grouped_split.csv"
)


# 读取文件
sample_metadata = pd.read_csv(SAMPLE_METADATA_PATH)
all_morgan_features = pd.read_csv(MORGAN_FEATURES_PATH)
split_data = pd.read_csv(SPLIT_PATH)


print("1. 样本信息")
print(f"形状：{sample_metadata.shape}")
print(f"列名：{sample_metadata.columns.tolist()}")
print(sample_metadata.head(3).to_string(index=False))


print("\n2. Morgan指纹")
print(f"形状：{all_morgan_features.shape}")
print(
    "前5个列名：",
    all_morgan_features.columns[:5].tolist(),
)
print(
    "后5个列名：",
    all_morgan_features.columns[-5:].tolist(),
)


print("\n3. 训练/测试划分")
print(f"形状：{split_data.shape}")
print(f"列名：{split_data.columns.tolist()}")
print(split_data.head(5).to_string(index=False))

1. 样本信息
形状：(1035, 8)
列名：['sample_id', 'name', 'CID', 'CAS', 'SMILES', 'canonical_smiles', 'label', 'fragment_count']
 sample_id             name   CID        CAS           SMILES canonical_smiles  label  fragment_count
         0 Ethanedioic acid   971   144-62-7     O=C(O)C(=O)O     O=C(O)C(=O)O      0               1
         1      Para-cymene  7463    99-87-6 Cc1ccc(C(C)C)cc1 Cc1ccc(C(C)C)cc1      0               1
         2       Kieselguhr 24261 61790-53-2         O=[Si]=O         O=[Si]=O      0               1

2. Morgan指纹
形状：(1035, 2049)
前5个列名： ['sample_id', 'MorganBit_0', 'MorganBit_1', 'MorganBit_2', 'MorganBit_3']
后5个列名： ['MorganBit_2043', 'MorganBit_2044', 'MorganBit_2045', 'MorganBit_2046', 'MorganBit_2047']

3. 训练/测试划分
形状：(1035, 7)
列名：['sample_id', 'label', 'fingerprint_hash', 'dataset_split', 'random_state', 'selected_fold', 'split_method']
 sample_id  label     fingerprint_hash dataset_split  random_state  selected_fold                            split_method
        

In [10]:
# 模型包中记录的预期训练样本数
expected_training_count = model_bundle["training_sample_count"]

# 检查保存的指纹列是否与模型特征顺序完全相同
saved_fingerprint_columns = [
    column
    for column in all_morgan_features.columns
    if column != "sample_id"
]

if saved_fingerprint_columns != feature_columns:
    raise ValueError(
        "Morgan指纹文件的特征名称或顺序与模型包不一致"
    )


# 只提取训练集样本编号
training_split = split_data.loc[
    split_data["dataset_split"] == "train",
    ["sample_id"],
].copy()


# 连接训练样本的名称和规范SMILES
training_reference = training_split.merge(
    sample_metadata[
        ["sample_id", "name", "canonical_smiles"]
    ],
    on="sample_id",
    how="inner",
    validate="one_to_one",
)


# 再连接训练样本的2048位Morgan指纹
training_reference = training_reference.merge(
    all_morgan_features[
        ["sample_id"] + feature_columns
    ],
    on="sample_id",
    how="inner",
    validate="one_to_one",
)


# 检查训练样本数量
if len(training_reference) != expected_training_count:
    raise ValueError(
        f"训练参考集有 {len(training_reference)} 个样本，"
        f"模型包记录的是 {expected_training_count} 个"
    )


# 检查是否意外包含测试样本
test_sample_ids = set(
    split_data.loc[
        split_data["dataset_split"] == "test",
        "sample_id",
    ]
)

reference_sample_ids = set(training_reference["sample_id"])

overlap_with_test = reference_sample_ids & test_sample_ids

if overlap_with_test:
    raise ValueError(
        f"训练参考集意外包含 {len(overlap_with_test)} 个测试样本"
    )


print("训练集相似度参考表构建成功")
print(
    "原始划分数量：",
    split_data["dataset_split"].value_counts().to_dict(),
)
print(f"训练参考样本数：{len(training_reference)}")
print(f"测试集重叠数：{len(overlap_with_test)}")
print(f"参考表形状：{training_reference.shape}")

print("\n前3个训练参考分子：")
print(
    training_reference[
        ["sample_id", "name", "canonical_smiles"]
    ]
    .head(3)
    .to_string(index=False)
)

训练集相似度参考表构建成功
原始划分数量： {'train': 827, 'test': 208}
训练参考样本数：827
测试集重叠数：0
参考表形状：(827, 2051)

前3个训练参考分子：
 sample_id         name canonical_smiles
         1  Para-cymene Cc1ccc(C(C)C)cc1
         2   Kieselguhr         O=[Si]=O
         3 Benzoic acid   O=C(O)c1ccccc1


In [11]:
# 将训练集的指纹表转换为NumPy数组
training_fingerprint_matrix = training_reference[
    feature_columns
].to_numpy(dtype=np.uint8)


# 将每个训练分子的0/1数组转换为RDKit位指纹
training_bit_vectors = []

for fingerprint_row in training_fingerprint_matrix:
    bit_vector = DataStructs.ExplicitBitVect(fp_size)

    # 找出值为1的指纹位
    active_bit_indices = np.flatnonzero(fingerprint_row)

    # 在RDKit位指纹中打开相应位置
    for bit_index in active_bit_indices:
        bit_vector.SetBit(int(bit_index))

    training_bit_vectors.append(bit_vector)


# 一次计算新分子与全部训练分子的Tanimoto相似度
tanimoto_similarities = np.asarray(
    DataStructs.BulkTanimotoSimilarity(
        morgan_fingerprint,
        training_bit_vectors,
    ),
    dtype=float,
)


# 基本安全检查
if len(tanimoto_similarities) != len(training_reference):
    raise ValueError("相似度数量与训练参考样本数量不一致")

if not np.all(
    (tanimoto_similarities >= 0)
    & (tanimoto_similarities <= 1)
):
    raise ValueError("发现超出0到1范围的Tanimoto相似度")


# 找出相似度最高的训练样本位置
nearest_position = int(
    np.argmax(tanimoto_similarities)
)

highest_tanimoto_similarity = float(
    tanimoto_similarities[nearest_position]
)

nearest_training_compound = training_reference.iloc[
    nearest_position
]


print("最高Tanimoto相似度计算完成")
print(
    f"最高Tanimoto相似度："
    f"{highest_tanimoto_similarity:.3f}"
)
print(
    f"最相似训练化合物："
    f"{nearest_training_compound['name']}"
)
print(
    f"最相似化合物的规范SMILES："
    f"{nearest_training_compound['canonical_smiles']}"
)
print(
    f"训练样本编号："
    f"{nearest_training_compound['sample_id']}"
)

最高Tanimoto相似度计算完成
最高Tanimoto相似度：0.357
最相似训练化合物：1-decanol
最相似化合物的规范SMILES：CCCCCCCCCCO
训练样本编号：53


In [12]:
# 初步的展示性谨慎提示线
# 它不是经过独立验证的正式适用域阈值
SIMILARITY_CAUTION_THRESHOLD = 0.40


if highest_tanimoto_similarity < SIMILARITY_CAUTION_THRESHOLD:
    similarity_message = (
        "结构相似性提示：该分子与训练集分子的最高相似度较低，"
        "模型预测需要谨慎解释。"
    )
else:
    similarity_message = (
        "结构相似性提示：训练集中存在与该分子具有一定结构相似性的样本，"
        "但结构相似并不能保证预测正确。"
    )


print("结构相似性检查")
print(
    f"最高Tanimoto相似度："
    f"{highest_tanimoto_similarity:.3f}"
)
print(
    f"最相似训练化合物："
    f"{nearest_training_compound['name']}"
)
print(similarity_message)

print("\n重要说明：")
print(
    "0.40仅作为初步展示性提示线，"
    "不是经过性能验证的正式适用域阈值。"
)
print(
    "Tanimoto相似度只反映当前Morgan指纹下的结构接近程度，"
    "不代表预测正确率或实验毒性相似度。"
)

结构相似性检查
最高Tanimoto相似度：0.357
最相似训练化合物：1-decanol
结构相似性提示：该分子与训练集分子的最高相似度较低，模型预测需要谨慎解释。

重要说明：
0.40仅作为初步展示性提示线，不是经过性能验证的正式适用域阈值。
Tanimoto相似度只反映当前Morgan指纹下的结构接近程度，不代表预测正确率或实验毒性相似度。
